在深度学习的发展历程中，研究人员从**绝对位置、相对位置、复数空间、连续隐空间**等多个维度探索了如何为模型注入顺序信息。

以下是深度学习领域中几乎所有主流及重要边缘化的位置编码（Positional Encoding/Embedding）方案的系统性汇总：

---

### 一、 绝对位置编码 (Absolute Positional Encodings)

这类方法直接为每个绝对坐标（如第 $t$ 个 Token）分配一个独立的向量。

1. **Sinusoidal Positional Encoding (正弦/余弦固定编码)**
* **来源**：Transformer 奠基论文 *Attention Is All You Need* (2017)。
* **原理**：使用不同频率的 $\sin$ 和 $\cos$ 函数组合生成固定的高维向量。


2. **Learned Absolute Position Embedding (可学习绝对位置嵌入)**
* **来源**：BERT, GPT 系列 (2018)。
* **原理**：将位置视作离散 ID（0, 1, 2...），初始化一个类似词表的可学习矩阵 `[Max_Seq_Len, d_model]`，通过反向传播训练。


3. **Floater (连续绝对位置编码)**
* **来源**：由刘一佳等提出 (2020)。
* **原理**：用神经网络（如常微分方程 ODE 或递归网络）将绝对位置建模为连续空间中的轨迹，允许对未见过的绝对位置进行内插和外推。



---

### 二、 相对位置编码 (Relative Positional Encodings)

这类方法不关心具体的绝对坐标，而是显式或隐式地对 Token 之间的相对距离（$i - j$）进行建模。

4. **Shaw's Relative Position Embedding (Shaw 相对位置)**
* **来源**：*Self-Attention with Relative Position Representations* (2018)。
* **原理**：在计算 Attention 的 $Q K^T$ 和值向量 $V$ 时，分别加入可学习的相对位置截断向量 $a_{ij}^K$ 和 $a_{ij}^V$。限制最大相对距离为 $k$（超过 $k$ 的距离共享同一个编码）。


5. **Transformer-XL PE (截断式相对位置)**
* **来源**：Transformer-XL (2019)。
* **原理**：为了支持跨片段（Segment）的循环机制，将 $Q K^T$ 展开，把绝对位置项替换为正弦编码的相对距离向量 $R_{i-j}$，并引入了两个全局偏置向量 $u$ 和 $v$。


6. **T5 Relative Bias (T5 相对位置偏置)**
* **来源**：Google T5 模型 (2019)。
* **原理**：最简化的相对位置设计。不在 Embedding 层做任何操作，而是直接在计算好的 Attention 分数矩阵 $A_{ij}$ 上加上一个标量偏置 $b_{i-j}$。偏置通过分桶（Bucket）机制实现非线性缩放（距离越近分辨越精细，距离越远桶越粗）。


7. **DeBERTa Relative PE (解耦注意力相对位置)**
* **来源**：微软 DeBERTa (2020)。
* **原理**：将内容（Content）和位置（Position）完全解耦。计算 Attention 分数时拆分为四项：内容-内容、内容-位置、位置-内容、位置-位置，均采用相对位置矩阵。


8. **KERPLE (Kernelized Relative Position Embedding)**
* **来源**：*Kernelized Relative Positional Embedding for Long Sequences* (2022)。
* **原理**：用核函数（如高斯核、拉普拉斯核）来数学化地显式建模相对位置的衰减趋势。



---

### 三、 混合与跨空间位置编码 (Hybrid & Mathematical Space PEs)

利用复数空间、矩阵旋转或几何变换，兼顾绝对位置的实现便利性与相对位置的数学优良性。

9. **RoPE (Rotary Position Embedding, 旋转位置编码)**
* **来源**：苏剑林 *Roformer* (2021) / LLaMA 广泛采用。
* **原理**：将二维子空间中的向量乘以一个正交旋转矩阵，旋转角度与绝对位置成正比。在内积操作下天然转化为相对位置。


10. **Complex-linear PE (复数线性位置编码)**
* **来源**：ICLR 2020 *Encoding word order in complex embeddings*。
* **原理**：将普通词嵌入拓展到复数空间，每个词用复数表示，而位置信息则作为复数的相位角（Phase Angle）叠加到词向量上。


11. **XPos (Exponential Distance Decay RoPE)**
* **来源**：微软 *A Length-Extrapolatable Transformer* (2022)，常用于 Meta-Transformer。
* **原理**：在 RoPE 的基础上引入了一个指数衰减阻尼因子，使得模型在旋转的基础上，随着相对距离增加，注意力权重呈现平滑的指数级衰减，大幅增强了长文本外推性。



---

### 四、 现代大模型长文本外推/插值位置编码 (Extrapolation & Interpolation PEs)

当模型需要处理远超训练长度（例如从 4k 扩展到 128k 甚至 1M）的文本时，专门针对 RoPE 进行修正的变体方案：

12. **Linear Interpolation / Position Interpolation (PI, 位置插值)**
* **来源**：Meta (2023)。
* **原理**：直接将超出训练长度的位置等比例“压缩”（插值）到原有的训练窗口内（如位置 $pos \to pos / N$）。


13. **NTK-aware Scaled RoPE**
* **来源**：开源社区开源贡献者 / 随后被各个大模型采纳。
* **原理**：基于神经切线核（NTK）理论，对高频和低频维度进行非均匀缩放。高频部分（微观位置）不插值保持精度，低频部分（宏观位置）进行插值，避免了低频信息的模糊。


14. **Dynamic NTK / YaRN (Yet another RoPE extensioN)**
* **来源**：YaRN 团队 (2023)。
* **原理**：目前最先进的 RoPE 扩展方案之一。在 NTK 基础上加入注意力矩阵的修正因子（Attention Spreading Correction），解决插值后注意力分布变平淡、模型困惑度（PPL）上升的问题。


15. **SuMMER (Symmetric Multi-Resolution RoPE)**
* **来源**：针对超长上下文的变体。
* **原理**：利用多分辨率机制分配不同的频域旋转速度，专门适配百万级（1M+）上下文。



---

### 五、 无参数/隐式位置编码 (Parameter-free & Implicit PEs)

16. **No Positional Encoding (无位置编码 / NoPE)**
* **来源**：部分学者研究及特定架构（如带有特定 Padding 或 Casual Mask 的 Decoder-Only 模型）。
* **原理**：完全不加位置编码。某些研究表明，在 Decoder-only 架构中，由于因果掩码（Causal Mask）和 Padding 本身就打破了排列不变性，大模型仅靠 Mask 就能隐式学到部分位置和顺序信息。


17. **ALiBi (Attention with Linear Biases, 线性注意力偏置)**
* **来源**：*Train Short, Test Long* (2021)，广泛用于 Bloom 模型。
* **原理**：完全不向 Embedding 注入任何位置信息。而是在 Attention 计算时，直接对每一个 Head 的 $Q K^T$ 矩阵赋予一个随相对距离线性递减的惩罚项（$-m \cdot |i-j|$），其中 $m$ 是每个 Head 固定的超参数斜率。外推性能极强。


18. **Sandwich Positioning (三明治位置编码)**
* **来源**：CogView / 改进视觉与文本多模态对齐。
* **原理**：在 Transformer 的各个 Layer 之间交叉重复注入位置编码（Layer 输入加一次，Layer 内部再加一次），以防位置信息在深层网络中被稀释。



---

### 六、 2D / 3D / 多维位置编码 (Multidimensional PEs)

针对计算机视觉（ViT）、视频模型以及 3D 点云/科学计算的多维空间编码。

19. **2D Sinusoidal / Learned PE (2D 空间位置编码)**
* **来源**：ViT (Vision Transformer) / DETR。
* **原理**：将高度（X 轴）和宽度（Y 轴）分别计算一维的位置编码（各占 $d/2$ 维度），然后拼接（Concat）成一个完整的 2D 位置编码加到图像 Patch 嵌入上。


20. **3D Grid PE (3D 网格位置编码)**
* **来源**：视频 Transformer（如 Timesformer）或 3D 医疗图像、点云处理。
* **原理**：在 2D 的基础上引入时间轴（Time 轴）或深度轴（Z 轴），将 X, Y, T 三个维度的编码进行组合或加和。


21. **CP-Vis (Conditional Positional Encoding / CPE)**
* **来源**：*Conditional Positional Encodings for Vision Transformers* (2021)。
* **原理**：利用深度可分离卷积（Depth-wise Convolution）动态地根据输入的图像内容生成位置编码。它天然支持任意分辨率输入（因为卷积具有平移不变性和对尺度的自适应性）。


22. **RoPE-2D / Axial RoPE (轴向旋转位置编码)**
* **来源**：多模态大模型（如图像/视频大模型 Qwen-VL, CogVLM）。
* **原理**：将 RoPE 扩展到二维，对图像的行坐标和列坐标分别应用不同角度的旋转矩阵。